In [2]:
from pathlib import Path
from collections import defaultdict
import re
import torch
path = Path('../data/')
dataverse_path = path / 'dataverse_files'

partitions_list = [
    'partition3_instances',
]

groups = defaultdict(list)

# Enumerated in factors of 10, may have to tweak
def getClasses(df):
    df_y = pd.DataFrame(columns=["category","time"],index=pd.RangeIndex(df.last_valid_index()+1))
    for index, row in df.iterrows():
        if(row["BFLARE"] == 1.0):
            df_y.iloc[index]=[10,row.time]
        elif(row["CFLARE"] == 1.0):
            df_y.iloc[index]=[100,row.time]
        elif(row["MFLARE"] == 1.0):
            df_y.iloc[index]=[1000,row.time]
        elif(row["XFLARE"] == 1.0):
            df_y.iloc[index]=[10000,row.time]
        else:
            df_y.iloc[index]=[0,row.time]
    return df_y

def parse_meta(fname):
    base = fname.name
    
    flare = re.search(r'@(\d+)', base)
    ar    = re.search(r'ar(\d+)', base)
    
    if flare and ar:
        return f"{flare.group(1)}_{ar.group(1)}"
    return None

for partition_inst in partitions_list:

    partition_inst_path = dataverse_path / partition_inst
    partition_num = partition_inst.replace('_instances', '')
    partition_folder_path = partition_inst_path / partition_num

    folder_path = partition_folder_path / 'FL'

    if folder_path.exists():

        for file in folder_path.glob('*.csv'):
            
            key = parse_meta(file)

            if key:
                groups[key].append(file)

print("Number of flare/AR groups:", len(groups))

import pandas as pd

def load_and_merge(file_list):

    dfs = []
    
    for f in file_list:
        df = pd.read_csv(f, sep="\t")  
        df["Timestamp"] = pd.to_datetime(df["Timestamp"])
        df = df.rename(columns={"Timestamp": "time"})

        dfs.append(df)

    if len(dfs) == 0:
        return pd.DataFrame()

    df = pd.concat(dfs).sort_values("time")
    df = df.drop_duplicates("time")

    return df.reset_index(drop=True)

merged = {}

for key, flist in groups.items():

    df = load_and_merge(flist)

    if len(df) > 0:
        merged[key] = df
print("Merged timelines:", len(merged))
df_y = getClasses(df)
# Set index and resample if required
# For now just drop time
# df_y = df_y.set_index("time")
df_y = df_y.drop("time", axis=1)
df_y = df_y.astype("float64")

feature_cols = [ 'TOTUSJZ', 'TOTUSJH', 'TOTPOT', 'ABSNJZH', 'SAVNCPP', 'USFLUX',
       'MEANPOT', 'R_VALUE', 'SHRGT45']

import numpy as np

feature_cols = [
    'TOTUSJZ', 'TOTUSJH', 'TOTPOT', 'ABSNJZH',
    'SAVNCPP', 'USFLUX', 'MEANPOT', 'R_VALUE', 'SHRGT45'
]

def transform(df):

    X = df[feature_cols].values.astype("float64")

    # stabilise scale
    X = np.log1p(np.clip(X, 0, None))

    return X

def split_time(X, frac=0.7):

    T = len(X)
    s = int(T * frac)

    return X[:s], X[s:]

def resample_12min(df):
    df = df.set_index("time")
    df_feat = df[feature_cols].resample("12min").mean()
    # Fill gaps robustly
    df_feat = df_feat.interpolate(limit_direction="both")
    df_feat = df_feat.ffill().bfill()  # catch remaining edge NaNs

    return df_feat.reset_index()

train_list = []
test_list  = []

 

for key, df in merged.items():

    df = resample_12min(df)
    X  = transform(df)

    if len(X) < 30:   # skip tiny timelines
        continue

    X_train, X_test = split_time(X)
    Y_train, Y_test = split_time(df_y)
    train_list.append(X_train)
    test_list.append(X_test)
print("Timelines used:", len(train_list))


def pad_numpy(seq_list):

    max_len = max(len(x) for x in seq_list)
    F = seq_list[0].shape[1]

    padded = np.zeros((len(seq_list), max_len, F))
    mask   = np.zeros((len(seq_list), max_len))

    for i, x in enumerate(seq_list):
        L = len(x)
        padded[i, :L] = x
        mask[i, :L]   = 1

    return padded, mask

train_pad, train_mask = pad_numpy(train_list)
test_pad,  test_mask  = pad_numpy(test_list)

mask_expanded = train_mask[..., None]

den = mask_expanded.sum(axis=(0, 1))              # (F,)
den = np.maximum(den, 1.0)

mu  = (train_pad * mask_expanded).sum(axis=(0, 1)) / den
var = (((train_pad - mu.reshape(1,1,-1))**2) * mask_expanded).sum(axis=(0, 1)) / den
sd  = np.sqrt(var) + 1e-6

mu = mu.reshape(1,1,-1)
sd = sd.reshape(1,1,-1)

train_z = (train_pad - mu) / sd
test_z  = (test_pad  - mu) / sd

# mark padded region as NaN so it never pollutes anything
train_z[train_mask == 0] = np.nan
test_z[test_mask == 0]   = np.nan

Number of flare/AR groups: 96
Merged timelines: 96
Timelines used: 96


In [ ]:
trainTensor_z = torch.from_numpy(train_z)
trainTensor_y = torch.from_numpy(Y_train.values)
testTensor_z = torch.from_numpy(test_z)
testTensor_Y = torch.from_numpy(Y_test.values)

class LSTM(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = torch.nn.LSTM(input_size=trainTensor_z[0][0].size()[0], hidden_size=1, dtype=torch.float64)
        self.linear = torch.nn.Linear(1, 1, dtype=torch.float64)
    def forward(self, x):
        x, _ = self.lstm(x)
        x = self.linear(x)
        return x

# TODO
# Determine hidden layer length
# Fix tensor broadcasting y = [122,1], x = [96,122,1]
# Deal with NaNs in dataset
# Loss function currently returning NaN, potentially as result of above


C:\Users\Flell\AppData\Local\Temp\ipykernel_21736\4165263593.py:2: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  trainTensor_y = torch.from_numpy(Y_train.values)


In [5]:
model = LSTM()
optimizer = torch.optim.Adam(model.parameters())
loss_fn = torch.nn.MSELoss()

 
n_epochs = 1000
for epoch in range(n_epochs):
    model.train()
    y_pred = model(trainTensor_z)
    loss = loss_fn(y_pred, trainTensor_y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()